<a href="https://colab.research.google.com/github/PendlimarriSivasankar/PS/blob/main/Pyspark_Tasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget -q https://github.com/saiadityaus1/test1/raw/main/df.csv -O df.csv
!wget -q https://github.com/saiadityaus1/test1/raw/main/df1.csv -O df1.csv
!wget -q https://github.com/saiadityaus1/test1/raw/main/dt.txt -O dt.txt
!wget -q https://github.com/saiadityaus1/test1/raw/main/file1.txt -O file1.txt
!wget -q https://github.com/saiadityaus1/test1/raw/main/file2.txt -O file2.txt
!wget -q https://github.com/saiadityaus1/test1/raw/main/file3.txt -O file3.txt
!wget -q https://github.com/saiadityaus1/test1/raw/main/file4.json -O file4.json
!wget -q https://github.com/saiadityaus1/test1/raw/main/file5.parquet -O file5.parquet
!wget -q https://github.com/saiadityaus1/test1/raw/main/file6 -O file6
!wget -q https://github.com/saiadityaus1/test1/raw/main/prod.csv -O prod.csv
!wget -q https://github.com/saiadityaus1/test1/raw/main/state.txt -O state.txt
!wget -q https://github.com/saiadityaus1/test1/raw/main/usdata.csv -O usdata.csv
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!pip install -q findspark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
import findspark
findspark.init()
import sys
import os
python_path = sys.executable
os.environ['PYSPARK_PYTHON'] = python_path
from pyspark import SparkConf
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
conf = SparkConf().setAppName("pyspark").setMaster("local[*]").set("spark.driver.host","localhost").set("spark.driver.allowMultipleContexts","true")
sc = SparkContext(conf=conf)
spark = SparkSession.builder.config("spark.driver.allowMultipleContexts","true").getOrCreate()
print("DONE==== START YOUR WORK👇 ")

DONE==== START YOUR WORK👇 


In [ ]:
# Pandas Schema

import pandas as pd

data = [['0', 'Y', 'N'], ['1', 'Y', 'Y'], ['2', 'N', 'Y'], ['3', 'Y', 'Y'], ['4', 'N', 'N']]
products = pd.DataFrame(data, columns=['product_id', 'low_fats', 'recyclable']).astype(
    {'product_id': 'int64', 'low_fats': 'category', 'recyclable': 'category'})
#converting to spark dataframe
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

products_df = spark.createDataFrame(products)
products_df.printSchema()

root
 |-- product_id: long (nullable = true)
 |-- low_fats: string (nullable = true)
 |-- recyclable: string (nullable = true)



In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Initialize Spark session
spark = SparkSession.builder.appName("FilterProducts").getOrCreate()

# Sample data
data = [
    (0, 'Y', 'N'),
    (1, 'Y', 'Y'),
    (2, 'N', 'Y'),
    (3, 'Y', 'Y'),
    (4, 'N', 'N')
]

# Define schema
columns = ["product_id", "low_fats", "recyclable"]

# Create DataFrame
df = spark.createDataFrame(data, columns)
df.show()
# Filter where low_fats = 'Y' and recyclable = 'Y'
filtered_df = df.filter((col("low_fats") == 'Y') & (col("recyclable") == 'Y'))
filtered_df.show()
# Select only product_id column
result_df = filtered_df.select("product_id")

# Show result
result_df.show()


+----------+--------+----------+
|product_id|low_fats|recyclable|
+----------+--------+----------+
|         0|       Y|         N|
|         1|       Y|         Y|
|         2|       N|         Y|
|         3|       Y|         Y|
|         4|       N|         N|
+----------+--------+----------+

+----------+--------+----------+
|product_id|low_fats|recyclable|
+----------+--------+----------+
|         1|       Y|         Y|
|         3|       Y|         Y|
+----------+--------+----------+

+----------+
|product_id|
+----------+
|         1|
|         3|
+----------+



In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import length, col

# Create Spark session
spark = SparkSession.builder.appName("TweetsFilter").getOrCreate()

# Sample data
data = [
    (1, "Let us Code"),
    (2, "More than fifteen chars are here!")
]
columns = ["tweet_id", "content"]

# Create DataFrame
tweets_df = spark.createDataFrame(data, columns)

# Filter where content length > 15
result_df = tweets_df.filter(length(col("content")) > 15) \
                     .select("tweet_id")

result_df.show()


+--------+
|tweet_id|
+--------+
|       2|
+--------+



In [1]:
from pyspark.sql.functions import length
from pyspark.sql import SparkSession
from pyspark.sql.functions import length, col

# Create Spark session
spark = SparkSession.builder.appName("TweetsFilter").getOrCreate()

# Sample data
data = [
    (1, "Let us Code"),
    (2, "More than fifteen chars are here!")
]
columns = ["tweet_id", "content"]

# Create DataFrame
tweets_df = spark.createDataFrame(data, columns)


# Register DataFrame as SQL temporary view
tweets_df.createOrReplaceTempView("Tweets")

# SQL query
sql_query = """
SELECT tweet_id
FROM Tweets
WHERE LENGTH(content) > 15
"""

result_sql_df = spark.sql(sql_query)
result_sql_df.show()

+--------+
|tweet_id|
+--------+
|       2|
+--------+



In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Sample data
data = [
 (1, "2024-01-01", "C001", 100),
 (2, "2024-01-02", "C001", 200),
 (3, "2024-01-03", "C001", 150),
 (4, "2024-01-04", "C002", 300),
 (5, "2024-01-05", "C002", 100)
]

columns = ["SaleID", "SaleDate", "CustomerID", "Amount"]

spark = SparkSession.builder.appName("RunningTotal").getOrCreate()
df = spark.createDataFrame(data, columns)

# Window specification (for full running total)
windowSpec = Window.partitionBy("CustomerID").orderBy("SaleDate")

# Add running total column
df_running_total = df.withColumn("RunningTotal",F.sum("Amount").over(windowSpec))

df_running_total.show()

+------+----------+----------+------+------------+
|SaleID|  SaleDate|CustomerID|Amount|RunningTotal|
+------+----------+----------+------+------------+
|     1|2024-01-01|      C001|   100|         100|
|     2|2024-01-02|      C001|   200|         300|
|     3|2024-01-03|      C001|   150|         450|
|     4|2024-01-04|      C002|   300|         300|
|     5|2024-01-05|      C002|   100|         400|
+------+----------+----------+------+------------+



In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

# Initialize Spark
spark = SparkSession.builder.appName("FirstSalePerProduct").getOrCreate()

# Sample data
data = [
    (1, 100, 2008, 10, 5000),
    (2, 100, 2009, 12, 5000),
    (7, 200, 2011, 15, 9000)
]

columns = ["sale_id", "product_id", "year", "quantity", "price"]

# Create DataFrame
df = spark.createDataFrame(data, columns)
df.show()
# Define window spec
windowSpec = Window.partitionBy("product_id").orderBy("year")

# Add row number
df_ranked = df.withColumn("row_num", row_number().over(windowSpec))

# Filter first sale per product
df_first = df_ranked.filter("row_num = 1").select("product_id", "year", "quantity", "price") \
    .withColumnRenamed("year", "first_year")

df_first.show()

+-------+----------+----+--------+-----+
|sale_id|product_id|year|quantity|price|
+-------+----------+----+--------+-----+
|      1|       100|2008|      10| 5000|
|      2|       100|2009|      12| 5000|
|      7|       200|2011|      15| 9000|
+-------+----------+----+--------+-----+

+----------+----------+--------+-----+
|product_id|first_year|quantity|price|
+----------+----------+--------+-----+
|       100|      2008|      10| 5000|
|       200|      2011|      15| 9000|
+----------+----------+--------+-----+



+---+-------+--------+------+----------+-------+
| id|country|   state|amount|trans_date|  month|
+---+-------+--------+------+----------+-------+
|121|     US|approved|  1000|2018-12-18|2018-12|
|122|     US|declined|  2000|2018-12-19|2018-12|
|123|     US|approved|  2000|2019-01-01|2019-01|
|124|     DE|approved|  2000|2019-01-07|2019-01|
+---+-------+--------+------+----------+-------+

+-------+-------+-----------+------------------+--------------+---------------------+
|  month|country|trans_count|trans_total_amount|approved_count|approved_total_amount|
+-------+-------+-----------+------------------+--------------+---------------------+
|2018-12|     US|          2|              3000|             1|                 1000|
|2019-01|     US|          1|              2000|             1|                 2000|
|2019-01|     DE|          1|              2000|             1|                 2000|
+-------+-------+-----------+------------------+--------------+---------------------+



In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

# Initialize Spark
spark = SparkSession.builder.appName("FirstSalePerProduct").getOrCreate()

# Sample data
data = [
    (1, 100, 2008, 10, 5000),
    (2, 100, 2009, 12, 5000),
    (7, 200, 2011, 15, 9000)
]

columns = ["sale_id", "product_id", "year", "quantity", "price"]

# Create DataFrame
df = spark.createDataFrame(data, columns)
df.show()
# Define window spec
windowSpec = Window.partitionBy("product_id").orderBy("year")

# Add row number
df_ranked = df.withColumn("row_num", row_number().over(windowSpec))

# Filter first sale per product
df_first = df_ranked.filter("row_num = 1").select("product_id", "year", "quantity", "price") \
    .withColumnRenamed("year", "first_year")

df_first.show()

+-------+----------+----+--------+-----+
|sale_id|product_id|year|quantity|price|
+-------+----------+----+--------+-----+
|      1|       100|2008|      10| 5000|
|      2|       100|2009|      12| 5000|
|      7|       200|2011|      15| 9000|
+-------+----------+----+--------+-----+

+----------+----------+--------+-----+
|product_id|first_year|quantity|price|
+----------+----------+--------+-----+
|       100|      2008|      10| 5000|
|       200|      2011|      15| 9000|
+----------+----------+--------+-----+



In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, date_format, sum, count, when

# Initialize Spark
spark = SparkSession.builder.appName("MonthlyTransactionSummary").getOrCreate()

# Sample data
data = [
    (121, "US", "approved", 1000, "2018-12-18"),
    (122, "US", "declined", 2000, "2018-12-19"),
    (123, "US", "approved", 2000, "2019-01-01"),
    (124, "DE", "approved", 2000, "2019-01-07")
]

columns = ["id", "country", "state", "amount", "trans_date"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Extract month from trans_date
df = df.withColumn("month", date_format(to_date("trans_date"), "yyyy-MM"))
df.show()
# Aggregate
df_summary = df.groupBy("month", "country").agg(
    count("*").alias("trans_count"),
    sum("amount").alias("trans_total_amount"),
    sum(when(col("state") == "approved", 1).otherwise(0)).alias("approved_count"),
    sum(when(col("state") == "approved", col("amount")).otherwise(0)).alias("approved_total_amount")
)

df_summary.show()


+---+-------+--------+------+----------+-------+
| id|country|   state|amount|trans_date|  month|
+---+-------+--------+------+----------+-------+
|121|     US|approved|  1000|2018-12-18|2018-12|
|122|     US|declined|  2000|2018-12-19|2018-12|
|123|     US|approved|  2000|2019-01-01|2019-01|
|124|     DE|approved|  2000|2019-01-07|2019-01|
+---+-------+--------+------+----------+-------+

+-------+-------+-----------+------------------+--------------+---------------------+
|  month|country|trans_count|trans_total_amount|approved_count|approved_total_amount|
+-------+-------+-----------+------------------+--------------+---------------------+
|2018-12|     US|          2|              3000|             1|                 1000|
|2019-01|     US|          1|              2000|             1|                 2000|
|2019-01|     DE|          1|              2000|             1|                 2000|
+-------+-------+-----------+------------------+--------------+---------------------+



In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, date_format, sum, count, when

# Initialize Spark
spark = SparkSession.builder.appName("MonthlyTransactionSummary").getOrCreate()

# Sample data
data = [
    (121, "US", "approved", 1000, "2018-12-18"),
    (122, "US", "declined", 2000, "2018-12-19"),
    (123, "US", "approved", 2000, "2019-01-01"),
    (124, "DE", "approved", 2000, "2019-01-07")
]

columns = ["id", "country", "state", "amount", "trans_date"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Extract month from trans_date
df = df.withColumn("month", date_format(to_date("trans_date"), "yyyy-MM"))
df.show()
# Aggregate
df_summary = df.groupBy("month", "country").agg(
    count("*").alias("trans_count"),
    sum("amount").alias("trans_total_amount"),
    sum(when(col("state") == "approved", 1).otherwise(0)).alias("approved_count"),
    sum(when(col("state") == "approved", col("amount")).otherwise(0)).alias("approved_total_amount")
)

df_summary.show()


+---+-------+--------+------+----------+-------+
| id|country|   state|amount|trans_date|  month|
+---+-------+--------+------+----------+-------+
|121|     US|approved|  1000|2018-12-18|2018-12|
|122|     US|declined|  2000|2018-12-19|2018-12|
|123|     US|approved|  2000|2019-01-01|2019-01|
|124|     DE|approved|  2000|2019-01-07|2019-01|
+---+-------+--------+------+----------+-------+

+-------+-------+-----------+------------------+--------------+---------------------+
|  month|country|trans_count|trans_total_amount|approved_count|approved_total_amount|
+-------+-------+-----------+------------------+--------------+---------------------+
|2018-12|     US|          2|              3000|             1|                 1000|
|2019-01|     US|          1|              2000|             1|                 2000|
|2019-01|     DE|          1|              2000|             1|                 2000|
+-------+-------+-----------+------------------+--------------+---------------------+



In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, to_date, sum, row_number
from pyspark.sql.window import Window

# Initialize Spark
spark = SparkSession.builder.appName("Day27Challenge").getOrCreate()

# Sample data
data = [
    (1001,"C001","2025-01-10","Shoes",2,2000,0.10),
    (1002,"C002","2025-01-11","Shirt",1,1200,None),
    (1003,"C003","2025-01-12","Shoes",1,2000,0.05),
    (1004,"C001","2025-01-13","Jeans",2,2500,0.15),
    (1005,"C002","2025-01-13","Shoes",1,2000,0.00),
    (1005,"C002","2025-01-13","Shoes",1,2000,0.00),  # duplicate
    (1006,"C004","2025-01-14","Shirt",None,1200,0.20),
    (1007,"C003","2025-01-15","Shoes",2,2000,0.10),
    (1008,"C005","2025-01-16","Shoes",1,2000,"Returned"),
]

cols = ["OrderID","CustomerID","OrderDate","Product","Quantity","UnitPrice","Discount"]

df = spark.createDataFrame(data, cols)
df.show()
# Remove duplicates
df_clean = df.dropDuplicates()
# Replace null discounts with 0.0
df_clean = df_clean.withColumn("Discount", when(col("Discount").isNull(), 0.0).otherwise(col("Discount")))
# Convert OrderDate to date type
df_clean = df_clean.withColumn("OrderDate", to_date("OrderDate"))
# Handle "Returned" rows
df_clean = df_clean.withColumn("SalesAmount", when(col("Discount") == "Returned",
    -1 * col("UnitPrice") * col("Quantity")).otherwise(None))
df_clean.show()

df_valid = df_clean.withColumn("SalesAmount", when(col("SalesAmount").isNull(),
    col("Quantity") * col("UnitPrice") * (1 - col("Discount"))).otherwise(col("SalesAmount")))


df_valid.groupBy("Product").agg(sum("SalesAmount").alias("TotalRevenue")).show()


df_valid.groupBy("CustomerID").agg(sum("SalesAmount").alias("TotalSpent")) \
    .orderBy(col("TotalSpent").desc()).limit(2).show()


windowSpec = Window.partitionBy("Product").orderBy("OrderDate").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_shoes = df_valid.filter(col("Product") == "Shoes") \
    .withColumn("RunningTotal", sum("SalesAmount").over(windowSpec)) \
    .select("OrderDate", "CustomerID", "SalesAmount", "RunningTotal")

df_shoes.orderBy("OrderDate").show()


+-------+----------+----------+-------+--------+---------+--------+
|OrderID|CustomerID| OrderDate|Product|Quantity|UnitPrice|Discount|
+-------+----------+----------+-------+--------+---------+--------+
|   1001|      C001|2025-01-10|  Shoes|       2|     2000|     0.1|
|   1002|      C002|2025-01-11|  Shirt|       1|     1200|    NULL|
|   1003|      C003|2025-01-12|  Shoes|       1|     2000|    0.05|
|   1004|      C001|2025-01-13|  Jeans|       2|     2500|    0.15|
|   1005|      C002|2025-01-13|  Shoes|       1|     2000|     0.0|
|   1005|      C002|2025-01-13|  Shoes|       1|     2000|     0.0|
|   1006|      C004|2025-01-14|  Shirt|    NULL|     1200|     0.2|
|   1007|      C003|2025-01-15|  Shoes|       2|     2000|     0.1|
|   1008|      C005|2025-01-16|  Shoes|       1|     2000|Returned|
+-------+----------+----------+-------+--------+---------+--------+

+-------+----------+----------+-------+--------+---------+--------+-----------+
|OrderID|CustomerID| OrderDate|Prod

In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum

spark = SparkSession.builder.getOrCreate()

data = [
 (1, "P001", "Laptop", 2, 600),
 (2, "P002", "Phone", 3, 300),
 (3, "P001", "Laptop", 1, 600),
 (4, "P003", "Tablet", 5, 150),
 (5, "P002", "Phone", 1, 300),
]

columns = ["transaction_id", "product_id", "product_name", "quantity", "price_per_unit"]
df = spark.createDataFrame(data, columns)

# Calculate total_sale per transaction
df = df.withColumn("total_sale", col("quantity") * col("price_per_unit"))
df.show()
# Aggregate and filter
result_df = (df.groupBy("product_id", "product_name")
 .agg(spark_sum("total_sale").alias("total_sale"))
 .filter(col("total_sale") > 1000))
result_df.show()

+--------------+----------+------------+--------+--------------+----------+
|transaction_id|product_id|product_name|quantity|price_per_unit|total_sale|
+--------------+----------+------------+--------+--------------+----------+
|             1|      P001|      Laptop|       2|           600|      1200|
|             2|      P002|       Phone|       3|           300|       900|
|             3|      P001|      Laptop|       1|           600|       600|
|             4|      P003|      Tablet|       5|           150|       750|
|             5|      P002|       Phone|       1|           300|       300|
+--------------+----------+------------+--------+--------------+----------+

+----------+------------+----------+
|product_id|product_name|total_sale|
+----------+------------+----------+
|      P002|       Phone|      1200|
|      P001|      Laptop|      1800|
+----------+------------+----------+



In [22]:
from pyspark.sql import SparkSession

# Sample data with nulls
data = [("Ravi", 28, None),
 ("Sunita", None, 55000),
 ("Amit", 35, 62000),
 (None, 42, 80000)]
columns = ["name", "age", "salary"]
df = spark.createDataFrame(data, columns)
df.show()
# A simple and safe way to clean
clean_df = df.na.fill({
 "age": 0,
 "salary": 0,
 "name": "Unknown"
})

clean_df.show()


+------+----+------+
|  name| age|salary|
+------+----+------+
|  Ravi|  28|  NULL|
|Sunita|NULL| 55000|
|  Amit|  35| 62000|
|  NULL|  42| 80000|
+------+----+------+

+-------+---+------+
|   name|age|salary|
+-------+---+------+
|   Ravi| 28|     0|
| Sunita|  0| 55000|
|   Amit| 35| 62000|
|Unknown| 42| 80000|
+-------+---+------+



In [23]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

# Initialize Spark
spark = SparkSession.builder.appName("FlightPath").getOrCreate()

# Sample data
data = [
    ("A", "India", "USA"),
    ("A", "USA", "UK"),
    ("A", "UK", "France"),
    ("B", "Japan", "Singapore"),
    ("B", "Singapore", "Australia")
]

columns = ["user_id", "flight_started_country", "flight_ended_country"]

df = spark.createDataFrame(data, columns)
df.show()
# Add row numbers to identify first and last flights
windowSpecAsc = Window.partitionBy("user_id").orderBy("flight_started_country")
windowSpecDesc = Window.partitionBy("user_id").orderBy(df.flight_started_country.desc())

df_origin = df.withColumn("rn", row_number().over(windowSpecAsc)).filter("rn = 1") \
    .select("user_id", df.flight_started_country.alias("origin_country"))
df_origin.show()
df_dest = df.withColumn("rn", row_number().over(windowSpecDesc)).filter("rn = 1") \
    .select("user_id", df.flight_ended_country.alias("destination_country"))
df_dest.show()
# Join origin and destination
df_final = df_origin.join(df_dest, on="user_id")
df_final.show()


+-------+----------------------+--------------------+
|user_id|flight_started_country|flight_ended_country|
+-------+----------------------+--------------------+
|      A|                 India|                 USA|
|      A|                   USA|                  UK|
|      A|                    UK|              France|
|      B|                 Japan|           Singapore|
|      B|             Singapore|           Australia|
+-------+----------------------+--------------------+

+-------+--------------+
|user_id|origin_country|
+-------+--------------+
|      A|         India|
|      B|         Japan|
+-------+--------------+

+-------+-------------------+
|user_id|destination_country|
+-------+-------------------+
|      A|                 UK|
|      B|          Australia|
+-------+-------------------+

+-------+--------------+-------------------+
|user_id|origin_country|destination_country|
+-------+--------------+-------------------+
|      A|         India|                 UK|
|

In [27]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum

spark = SparkSession.builder.getOrCreate()
df = spark.read.parquet("/mnt/data/sales")
east_sales = df.filter(col("region") == "East") \
.groupBy("product") \
.agg(sum("amount").alias("total_sales"))
df = df.withColumn(
    "customer_segment",
    when(col("spending_score").isNull(), "Unknown")
    .when(col("spending_score") >= 80, "High Value")
    .when((col("spending_score") >= 50) & (col("spending_score") < 80), "Medium Value")
    .otherwise("Low Value")
)
df.groupBy("customer_segment").count().orderBy("count", ascending=False).show()


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/mnt/data/sales.

In [29]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode

spark = SparkSession.builder.getOrCreate()

data = [
("Alice", ["Laptop", "Mouse"]),
("Bob", ["Tablet"]),
("Charlie", ["Phone", "Charger", "Earbuds"])
]

df = spark.createDataFrame(data, ["customer_name", "products"])
df.show(truncate=False)

df_exploded = df.withColumn("product", explode(df.products))
df_exploded.show(truncate=False)

+-------------+-------------------------+
|customer_name|products                 |
+-------------+-------------------------+
|Alice        |[Laptop, Mouse]          |
|Bob          |[Tablet]                 |
|Charlie      |[Phone, Charger, Earbuds]|
+-------------+-------------------------+

+-------------+-------------------------+-------+
|customer_name|products                 |product|
+-------------+-------------------------+-------+
|Alice        |[Laptop, Mouse]          |Laptop |
|Alice        |[Laptop, Mouse]          |Mouse  |
|Bob          |[Tablet]                 |Tablet |
|Charlie      |[Phone, Charger, Earbuds]|Phone  |
|Charlie      |[Phone, Charger, Earbuds]|Charger|
|Charlie      |[Phone, Charger, Earbuds]|Earbuds|
+-------------+-------------------------+-------+



In [30]:
from pyspark import SparkContext
sc = SparkContext.getOrCreate()

acc = sc.accumulator(0)

rdd = sc.parallelize([1, 2, 3, 4, 5])
rdd.foreach(lambda x: acc.add(x))

print("Total Sum:", acc.value)  # Output: 15

lookup = {"A": "Apple", "B": "Banana", "C": "Cherry"}
bcast = sc.broadcast(lookup)

rdd = sc.parallelize(["A", "B", "C"])
print(rdd.map(lambda x: bcast.value[x]).collect())
# Output: ['Apple', 'Banana', 'Cherry']


Total Sum: 15
['Apple', 'Banana', 'Cherry']


In [36]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Initialize SparkSession with Delta Lake package
spark = SparkSession.builder \
    .appName("DeltaLakeExample") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()


new_data = spark.read.format("delta") \
    .load("/mnt/datalake/transactions") \
    .where(col("_ingestion_date") >= "2025-08-23")

agg_data = new_data.groupBy("account_id") \
    .agg({"amount": "sum"}) \
    .withColumnRenamed("sum(amount)", "daily_spent")

agg_data.write.format("delta") \
    .mode("append") \
    .save("/mnt/datalake/gold/transactions_summary")


gold_table = DeltaTable.forPath(spark, "/mnt/datalake/gold/transactions_summary")

gold_table.alias("gold").merge(
    agg_data.alias("new"),
    "gold.account_id = new.account_id"
).whenMatchedUpdate(set={"daily_spent": "new.daily_spent"}) \
 .whenNotMatchedInsertAll() \
 .execute()

# NOTE: The following line assumes an 'etl_metadata' table exists with a 'last_run' column.
# If this table does not exist or has a different schema, this line will cause an error.
# You may need to adjust this based on your actual metadata tracking mechanism.
# last_load = spark.read.table("etl_metadata").filter("job_name = 'daily_txn'").select("last_run").collect()[0][0]

# new_data = spark.read.format("delta") \
#     .load("/mnt/datalake/transactions") \
#     .where(col("_ingestion_date") > last_load)

Py4JJavaError: An error occurred while calling o2028.load.
: java.util.ServiceConfigurationError: org.apache.spark.sql.sources.DataSourceRegister: Error accessing configuration file
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:582)
	at java.base/java.util.ServiceLoader$LazyClassPathLookupIterator.parse(ServiceLoader.java:1173)
	at java.base/java.util.ServiceLoader$LazyClassPathLookupIterator.nextProviderClass(ServiceLoader.java:1206)
	at java.base/java.util.ServiceLoader$LazyClassPathLookupIterator.hasNextService(ServiceLoader.java:1221)
	at java.base/java.util.ServiceLoader$LazyClassPathLookupIterator.hasNext(ServiceLoader.java:1265)
	at java.base/java.util.ServiceLoader$2.hasNext(ServiceLoader.java:1300)
	at java.base/java.util.ServiceLoader$3.hasNext(ServiceLoader.java:1385)
	at scala.collection.convert.Wrappers$JIteratorWrapper.hasNext(Wrappers.scala:45)
	at scala.collection.Iterator.foreach(Iterator.scala:943)
	at scala.collection.Iterator.foreach$(Iterator.scala:

In [34]:
!pip install delta-spark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 11.0 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.0.1-py2.py3-none-any.whl size=434813800 sha256=bce45a15ad45fe7349bd5f30e8a41118e3ed691f21005cf34809ad9356f296f6
  Stored in directory: /root/.cache/pip/wheels/31/9f/68/f89fb34ccd886909be7d0e390eaaf97f21efdf540c0ee8dbcd
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.7
    Uninstalling py4j-0.10.9.7:
      Successfully uninstalled py4j-0.10.9.7
  Attempting uninstall: pyspark
    Found existing installation: pyspark 3.5.1
    Uninstalling pyspark-3.5.1:
      Successfully uninstalled pyspark-3.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conn

In [38]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Initialize SparkSession with Delta Lake package
spark = SparkSession.builder \
    .appName("DeltaLakeExample") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()


new_data = spark.read.format("delta") \
    .load("/mnt/datalake/transactions") \
    .where(col("_ingestion_date") >= "2025-08-23")

agg_data = new_data.groupBy("account_id") \
    .agg({"amount": "sum"}) \
    .withColumnRenamed("sum(amount)", "daily_spent")

agg_data.write.format("delta") \
    .mode("append") \
    .save("/mnt/datalake/gold/transactions_summary")


gold_table = DeltaTable.forPath(spark, "/mnt/datalake/gold/transactions_summary")

gold_table.alias("gold").merge(
    agg_data.alias("new"),
    "gold.account_id = new.account_id"
).whenMatchedUpdate(set={"daily_spent": "new.daily_spent"}) \
 .whenNotMatchedInsertAll() \
 .execute()

# NOTE: The following line assumes an 'etl_metadata' table exists with a 'last_run' column.
# If this table does not exist or has a different schema, this line will cause an error.
# You may need to adjust this based on your actual metadata tracking mechanism.
# last_load = spark.read.table("etl_metadata").filter("job_name = 'daily_txn'").select("last_run").collect()[0][0]

# new_data = spark.read.format("delta") \
#     .load("/mnt/datalake/transactions") \
#     .where(col("_ingestion_date") > last_load)

Py4JJavaError: An error occurred while calling o2074.load.
: java.util.ServiceConfigurationError: org.apache.spark.sql.sources.DataSourceRegister: Error accessing configuration file
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:582)
	at java.base/java.util.ServiceLoader$LazyClassPathLookupIterator.parse(ServiceLoader.java:1173)
	at java.base/java.util.ServiceLoader$LazyClassPathLookupIterator.nextProviderClass(ServiceLoader.java:1206)
	at java.base/java.util.ServiceLoader$LazyClassPathLookupIterator.hasNextService(ServiceLoader.java:1221)
	at java.base/java.util.ServiceLoader$LazyClassPathLookupIterator.hasNext(ServiceLoader.java:1265)
	at java.base/java.util.ServiceLoader$2.hasNext(ServiceLoader.java:1300)
	at java.base/java.util.ServiceLoader$3.hasNext(ServiceLoader.java:1385)
	at scala.collection.convert.Wrappers$JIteratorWrapper.hasNext(Wrappers.scala:45)
	at scala.collection.Iterator.foreach(Iterator.scala:943)
	at scala.collection.Iterator.foreach$(Iterator.scala:

In [41]:
try:
    import dataproc_spark_connect
    print("dataproc-spark-connect is installed.")
    print("This package might be conflicting with your Spark setup after installing delta-spark.")
    print("Consider uninstalling it using: !pip uninstall dataproc-spark-connect")
except ImportError:
    print("dataproc-spark-connect is not installed.")

dataproc-spark-connect is not installed.


In [43]:
import pyspark
import delta

print(f"PySpark version: {pyspark.__version__}")
# Delta Lake's version is tied to the spark version
# You can check the delta-spark package version
from delta import __version__ as delta_version
print(f"delta-spark package version: {delta_version}")

PySpark version: 3.5.1


ImportError: cannot import name '__version__' from 'delta' (/usr/local/lib/python3.12/dist-packages/delta/__init__.py)

In [45]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Initialize SparkSession with Delta Lake package
spark = SparkSession.builder \
    .appName("DeltaLakeExample") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()


new_data = spark.read.format("delta") \
    .load("/mnt/datalake/transactions") \
    .where(col("_ingestion_date") >= "2025-08-23")

agg_data = new_data.groupBy("account_id") \
    .agg({"amount": "sum"}) \
    .withColumnRenamed("sum(amount)", "daily_spent")

agg_data.write.format("delta") \
    .mode("append") \
    .save("/mnt/datalake/gold/transactions_summary")


gold_table = DeltaTable.forPath(spark, "/mnt/datalake/gold/transactions_summary")

gold_table.alias("gold").merge(
    agg_data.alias("new"),
    "gold.account_id = new.account_id"
).whenMatchedUpdate(set={"daily_spent": "new.daily_spent"}) \
 .whenNotMatchedInsertAll() \
 .execute()

# NOTE: The following line assumes an 'etl_metadata' table exists with a 'last_run' column.
# If this table does not exist or has a different schema, this line will cause an error.
# You may need to adjust this based on your actual metadata tracking mechanism.
# last_load = spark.read.table("etl_metadata").filter("job_name = 'daily_txn'").select("last_run").collect()[0][0]

# new_data = spark.read.format("delta") \
#     .load("/mnt/datalake/transactions") \
#     .where(col("_ingestion_date") > last_load)

Py4JJavaError: An error occurred while calling o2556.load.
: java.util.ServiceConfigurationError: org.apache.spark.sql.sources.DataSourceRegister: Error accessing configuration file
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:582)
	at java.base/java.util.ServiceLoader$LazyClassPathLookupIterator.parse(ServiceLoader.java:1173)
	at java.base/java.util.ServiceLoader$LazyClassPathLookupIterator.nextProviderClass(ServiceLoader.java:1206)
	at java.base/java.util.ServiceLoader$LazyClassPathLookupIterator.hasNextService(ServiceLoader.java:1221)
	at java.base/java.util.ServiceLoader$LazyClassPathLookupIterator.hasNext(ServiceLoader.java:1265)
	at java.base/java.util.ServiceLoader$2.hasNext(ServiceLoader.java:1300)
	at java.base/java.util.ServiceLoader$3.hasNext(ServiceLoader.java:1385)
	at scala.collection.convert.Wrappers$JIteratorWrapper.hasNext(Wrappers.scala:45)
	at scala.collection.Iterator.foreach(Iterator.scala:943)
	at scala.collection.Iterator.foreach$(Iterator.scala:

In [47]:
!pip show delta-spark

Name: delta-spark
Version: 4.0.0
Summary: Python APIs for using Delta Lake with Apache Spark
Home-page: https://github.com/delta-io/delta/
Author: The Delta Lake Project Authors
Author-email: delta-users@googlegroups.com
License: Apache-2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: importlib-metadata, pyspark
Required-by: 


In [48]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

# Initialize Spark
spark = SparkSession.builder.appName("HighestPaidPerDepartment").getOrCreate()

# Sample data
data = [
    (1, "John", "HR", 3000),
    (2, "Mike", "IT", 4000),
    (3, "Sara", "HR", 3500),
    (4, "David", "Finance", 4500),
    (5, "Anna", "IT", 4200),
]
columns = ["emp_id", "name", "department", "salary"]

# Create DataFrame
df = spark.createDataFrame(data, columns)
df.show()
# Define window spec
windowSpec = Window.partitionBy("department").orderBy(df.salary.desc())

# Add row number and filter top earners
df_top = df.withColumn("rank", row_number().over(windowSpec)) \
           .filter("rank = 1") \
           .drop("rank")

df_top.show()


Py4JJavaError: An error occurred while calling o2824.showString.
: org.apache.spark.SparkException: Cannot find catalog plugin class for catalog 'spark_catalog': org.apache.spark.sql.delta.catalog.DeltaCatalog.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.catalogPluginClassNotFoundForCatalogError(QueryExecutionErrors.scala:1925)
	at org.apache.spark.sql.connector.catalog.Catalogs$.load(Catalogs.scala:70)
	at org.apache.spark.sql.connector.catalog.CatalogManager.loadV2SessionCatalog(CatalogManager.scala:67)
	at org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$v2SessionCatalog$2(CatalogManager.scala:86)
	at scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:86)
	at org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$v2SessionCatalog$1(CatalogManager.scala:86)
	at scala.Option.map(Option.scala:230)
	at org.apache.spark.sql.connector.catalog.CatalogManager.v2SessionCatalog(CatalogManager.scala:85)
	at org.apache.spark.sql.connector.catalog.Cata